In [8]:
import requests
import time
import pandas as pd
from datetime import *
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt

In [14]:
# 進入台灣銀行牌告匯率網頁，查看當日匯率資料
url = "https://rate.bot.com.tw/xrt?Lang=zh-TW"
resp = requests.get(url)
resp.encoding = 'utf-8'
# print(resp.text)  # 如有需要，列印出網頁原始碼

html = BeautifulSoup(resp.text, "lxml")
rate_table = html.find(name='table', attrs={'title':'牌告匯率'}).find(name='tbody').find_all(name='tr')

# 查詢美金(也就是匯率表的第一個元素)對台幣的匯率
currency = rate_table[0].find(name='div', attrs={'class':'visible-phone print_hide'})
print(currency.get_text().replace(" ", ""))  # 去掉空白

buy_rate = rate_table[0].find(name='td', attrs={'data-table':'本行現金買入'})
sell_rate = rate_table[0].find(name='td', attrs={'data-table':'本行現金賣出'})
print("即時現金買入: %s, 即時現金賣出: %s" % (buy_rate.get_text(), sell_rate.get_text()))



美金(USD)

即時現金買入: 31.96, 即時現金賣出: 32.63


In [20]:
# 先到牌告匯率首頁，爬取所有貨幣的種類
url = "https://rate.bot.com.tw/xrt?Lang=zh-TW"
resp = requests.get(url)
resp.encoding = 'utf-8'
html = BeautifulSoup(resp.text, "lxml")
rate_table = html.find(name='table', attrs={'title':'牌告匯率'}).find(name='tbody').find_all(name='tr')

# 擷取匯率表格，把美金(也就是匯率表的第一個元素)擷取出來，查詢其歷史匯率
currency = rate_table[0].find(name='div', attrs={'class':'visible-phone print_hide'})
print('1',currency.get_text().replace(" ", ""))  # 貨幣種類

buy_rate = rate_table[0].find(name='td', attrs={'data-table':'本行現金買入'})
sell_rate = rate_table[0].find(name='td', attrs={'data-table':'本行現金賣出'})
print('2',"即時現金買入: %s, 即時現金賣出: %s" % (buy_rate.get_text(), sell_rate.get_text()))
# 針對美金，找到其「歷史匯率」的首頁 
history_link = rate_table[0].find(name='td', attrs={'data-table':'歷史匯率'})
# history_rate_link = "https://rate.bot.com.tw" + history_link.a["href"]  # 該貨幣的歷史資料首頁
history_rate_link = "https://rate.bot.com.tw/xrt/quote/ltm/USD"
print('3',history_rate_link)

#
# 到貨幣歷史匯率網頁，選則該貨幣的「歷史區間」，送出查詢後，觀察其網址變化情形，再試著抓取其歷史匯率資料
#
# 用「quote/年-月」去取代網址內容，就可以連到該貨幣的歷史資料
quote_history_url = history_rate_link.replace("history", "quote/2019-08")
resp = requests.get(quote_history_url)
resp.encoding = 'utf-8'
history = BeautifulSoup(resp.text, "lxml")
history_table = history.find(name='table', attrs={'title':'歷史本行營業時間牌告匯率'}).find(name='tbody').find_all(name='tr')

#
# 擷取到歷史匯率資料後，把資料彙整起來並畫出趨勢圖
#
date_history = list()
history_buy = list()
history_sell = list()

for history_rate in history_table:
    # 擷取日期資料
    if history_rate.a is None:
        continue
    
    date_string = history_rate.a.get_text()
    date = datetime.strptime(date_string, '%Y/%m/%d').strftime('%Y/%m/%d')  # 轉換日期格式
    date_history.append(date)  # 日期歷史資料

    history_ex_rate = history_rate.find_all(name='td', attrs={'class':'rate-content-cash text-right print_table-cell'})
    history_buy.append(float(history_ex_rate[0].get_text()))  # 歷史買入匯率
    history_sell.append(float(history_ex_rate[1].get_text()))  # 歷史賣出匯率

# 將匯率資料建成dataframe形式
History_ExchangeRate = pd.DataFrame({'date': date_history,
                                    'buy_rate':history_buy,
                                    'sell_rate':history_sell})

History_ExchangeRate = History_ExchangeRate.set_index('date')  # 指定日期欄位為datafram的index
History_ExchangeRate = History_ExchangeRate.sort_index(ascending=True)

print('4', History_ExchangeRate)
# 畫出歷史匯率軌跡圖
# plt.figure(figsize=(10, 8))
# History_ExchangeRate[['buy_rate','sell_rate']].plot()  # x=['date'], y=[['buy_rate','sell_rate']] 
# plt.legend(loc="upper left")
# plt.show()


美金(USD)

即時現金買入: 31.945, 即時現金賣出: 32.615
            buy_rate  sell_rate
date                           
2023/07/03    30.735     31.405
2023/07/04    30.715     31.385
2023/07/05    30.750     31.420
2023/07/06    30.830     31.500
2023/07/07    30.935     31.605
...              ...        ...
2023/09/26    31.830     32.500
2023/09/27    31.845     32.515
2023/09/28    31.870     32.540
2023/10/02    31.850     32.520
2023/10/03    31.945     32.615

[67 rows x 2 columns]


In [31]:
# url = 'https://rate.bot.com.tw/xrt/flcsv/0/day'   # 牌告匯率 CSV 網址
url = 'https://rate.bot.com.tw/xrt/flcsv/0/l6m/USD'
response = requests.get(url)   # 爬取網址內容
response.encoding = 'utf-8'    # 調整回應訊息編碼為 utf-8，避免編碼不同造成亂碼
rt = response.text             # 以文字模式讀取內容
rts = rt.split('\n')       # 使用「換行」將內容拆分成串列
for i in rts:              # 讀取串列的每個項目
    try:                             # 使用 try 避開最後一行的空白行
        a = i.split(',')             # 每個項目用逗號拆分成子串列
        print(a[0] + ': ' + a[13])   # 取出第一個 ( 0 ) 和第十四個項目 ( 13 )
    except:
      break

In [8]:
url = 'https://rate.bot.com.tw/xrt/flcsv/0/l6m/USD'
response = requests.get(url)
response.encoding = 'utf-8'

lines = response.text.split('\n')[1:]

data = []
for line in lines:
    if line:  # Check if the line is not empty
        items = line.split(',')
        date = items[0]
        rate = items[13]
        data.append([date, rate])

df = pd.DataFrame(data, columns=['Date', 'USD_Rate'])
print(df)

         Date  USD_Rate
0    20231003  32.61500
1    20231002  32.52000
2    20230928  32.54000
3    20230927  32.51500
4    20230926  32.50000
..        ...       ...
122  20230412  30.77000
123  20230411  30.74500
124  20230410  30.71500
125  20230407  30.71000
126  20230406  30.79000

[127 rows x 2 columns]
